# Visualize Doctor Breath Labels

Load an exported breath-label `.pkl`, inspect annotator outputs, and visualize one breath at a time.

In [ ]:
from pathlib import Path

import ast
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

plt.rcParams["figure.figsize"] = (12, 4)

In [ ]:
# === CONFIGURATION ===
BASE_DIR = Path.cwd()
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

EXPORT_ROOT = BASE_DIR / "stored_results" / "05_breath_label_exports"
VERSION_TAG = "20260301"   # ← 변경: 분석할 버전 태그
PATIENT_ID = "03"          # ← 변경: 분석할 환자 ID
# =====================

PKL_PATH = EXPORT_ROOT / VERSION_TAG / f"patient_{PATIENT_ID}" / f"BB_patient_{PATIENT_ID}_clustered_breaths__doctor_labels.pkl"

if not PKL_PATH.exists():
    raise FileNotFoundError(f"Exported PKL not found: {PKL_PATH}")

df = pd.read_pickle(PKL_PATH)
doctor_cols = [c for c in df.columns if c.startswith("doctor__")]
label_cols = [c for c in doctor_cols if c.endswith("__label")]

print("Loaded:", PKL_PATH)
print("Shape:", df.shape)
print("Doctor label columns:", label_cols)
display(df.head())

In [ ]:
# Label distribution per annotator
summary_frames = []
for label_col in label_cols:
    annotator = label_col[len("doctor__") : -len("__label")]
    counts = (
        df[label_col]
        .fillna("UNLABELED")
        .replace("", "UNLABELED")
        .value_counts(dropna=False)
        .rename_axis("label")
        .reset_index(name="count")
    )
    counts.insert(0, "annotator", annotator)
    summary_frames.append(counts)

summary_df = pd.concat(summary_frames, ignore_index=True) if summary_frames else pd.DataFrame()
display(summary_df)

In [ ]:
# Anomaly labeling progress
anomaly_mask = df["AE_abnormal"].fillna(False).astype(bool)
anomaly_total = int(anomaly_mask.sum())

progress_rows = []
for label_col in label_cols:
    annotator = label_col[len("doctor__") : -len("__label")]
    labeled_mask = df[label_col].fillna("").astype(str).str.strip() != ""
    anomaly_labeled = int((anomaly_mask & labeled_mask).sum())
    progress_rows.append(
        {
            "annotator": annotator,
            "anomaly_total": anomaly_total,
            "anomaly_labeled": anomaly_labeled,
            "anomaly_remaining": anomaly_total - anomaly_labeled,
            "anomaly_label_rate": round(100.0 * anomaly_labeled / anomaly_total, 2) if anomaly_total else 0.0,
        }
    )

anomaly_progress_df = pd.DataFrame(progress_rows)
display(anomaly_progress_df)

In [ ]:
# Per-sample label comparison: breath_id × annotator
rename_map = {col: col[len("doctor__"):-len("__label")] for col in label_cols}
label_comparison_df = (
    df[["breath_id"] + label_cols]
    .rename(columns=rename_map)
    .set_index("breath_id")
)
labeled_mask = label_comparison_df.apply(lambda col: col.fillna("").astype(str).str.strip() != "").any(axis=1)
display(label_comparison_df[labeled_mask])

In [ ]:
# All patients summary across all exported PKLs
all_rows = []
for pkl_path in sorted(EXPORT_ROOT.glob("*/patient_*/BB_*__doctor_labels.pkl")):
    version = pkl_path.parts[-3]
    patient_dir = pkl_path.parts[-2]  # e.g. patient_03
    pid = patient_dir.replace("patient_", "")
    try:
        tmp = pd.read_pickle(pkl_path)
        tmp_label_cols = [c for c in tmp.columns if c.startswith("doctor__") and c.endswith("__label")]
        anomaly_m = tmp["AE_abnormal"].fillna(False).astype(bool)
        a_total = int(anomaly_m.sum())
        for lc in tmp_label_cols:
            ann = lc[len("doctor__") : -len("__label")]
            labeled_m = tmp[lc].fillna("").astype(str).str.strip() != ""
            a_labeled = int((anomaly_m & labeled_m).sum())
            all_rows.append({
                "version": version,
                "patient_id": pid,
                "annotator": ann,
                "anomaly_total": a_total,
                "anomaly_labeled": a_labeled,
                "anomaly_remaining": a_total - a_labeled,
                "label_rate_%": round(100.0 * a_labeled / a_total, 1) if a_total else 0.0,
            })
    except Exception as e:
        all_rows.append({"version": version, "patient_id": pid, "annotator": "ERROR", "anomaly_total": 0,
                         "anomaly_labeled": 0, "anomaly_remaining": 0, "label_rate_%": 0.0})

all_summary_df = pd.DataFrame(all_rows).sort_values(["version", "patient_id", "annotator"]).reset_index(drop=True)
print(f"총 {len(all_summary_df)}개 (version x patient x annotator) 조합")
display(all_summary_df)

In [ ]:
# Filter rows by a specific label
SELECTED_LABEL = "Phasic"  # change to: Sigh / Apnea / Hiccup / Tonic Burst / Crying / NeedSplit / NotSure

filtered = df.copy()
for label_col in label_cols:
    filtered = filtered[filtered[label_col].fillna("") == SELECTED_LABEL]

print("Filtered rows:", len(filtered))
display(filtered.head(20))

In [ ]:
def _to_signal_array(value):
    if isinstance(value, np.ndarray):
        return value.astype(float)
    if isinstance(value, list):
        return np.asarray(value, dtype=float)
    if isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)
        except Exception:
            return np.asarray([], dtype=float)
        if isinstance(parsed, list):
            return np.asarray(parsed, dtype=float)
    return np.asarray([], dtype=float)


ROW_INDEX = 1  # choose a row index from df.index or filtered.index
row = df.loc[ROW_INDEX]

orig = _to_signal_array(row.get("original_edi_signal"))
filt = _to_signal_array(row.get("filtered_edi_signal"))
peak_rel = row.get("peak_rel_indices", [])
if not isinstance(peak_rel, list):
    peak_rel = []

label_text = []
for label_col in label_cols:
    annotator = label_col[len("doctor__") : -len("__label")]
    label_val = row.get(label_col)
    if pd.notna(label_val) and str(label_val).strip():
        label_text.append(f"{annotator}: {label_val}")

fig, axes = plt.subplots(2, 1, sharex=True, figsize=(12, 6))

axes[0].plot(orig, color="black", linewidth=1.5)
axes[0].set_title(f"Original EDI | breath_id={row['breath_id']} | {' | '.join(label_text) if label_text else 'unlabeled'}")
axes[0].set_ylabel("original")

axes[1].plot(filt, color="tab:blue", linewidth=1.5)
if peak_rel:
    valid_peak_rel = [p for p in peak_rel if 0 <= int(p) < len(filt)]
    axes[1].scatter(valid_peak_rel, filt[valid_peak_rel], color="tab:orange", s=40, zorder=3)
axes[1].set_title("Filtered EDI with stored peak_rel_indices")
axes[1].set_ylabel("filtered")
axes[1].set_xlabel("sample index within breath")

plt.tight_layout()
plt.show()

display(row.to_frame().T)